In [3]:
import pandas as pd
import numpy as np


from sklearn.feature_extraction.text import TfidfVectorizer
import re


from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv')
y_test = pd.read_csv('y_test.csv')

print("Libraries and Structured Data Loaded Successfully!")

Libraries and Structured Data Loaded Successfully!


In [5]:
import re


def clean_text(text):
    if pd.isna(text): 
        return ""
    text = text.lower() 
    text = re.sub(r'[^a-z0-9\s]', '', text) 
    return text


sample_desc = "Mint condition! No scratches... 100% perfect."
cleaned_desc = clean_text(sample_desc)

print("Original:", sample_desc)
print("Cleaned:", cleaned_desc)

Original: Mint condition! No scratches... 100% perfect.
Cleaned: mint condition no scratches 100 perfect


In [6]:
import pandas as pd


try:
    df_raw = pd.read_csv('car data.csv')
    print("--- Columns in 'car data.csv' ---")
    print(df_raw.columns.tolist())
except Exception as e:
    print(f"Error loading car data.csv: {e}")

print("\n")


try:
    df_cleaned = pd.read_csv('cleaned_car_data.csv')
    print("--- Columns in 'cleaned_car_data.csv' ---")
    print(df_cleaned.columns.tolist())
except Exception as e:
    print(f"Error loading cleaned_car_data.csv: {e}")

--- Columns in 'car data.csv' ---
['Car_Name', 'Year', 'Selling_Price', 'Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner']


--- Columns in 'cleaned_car_data.csv' ---
['Car_Name', 'Year', 'Selling_Price', 'Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner']


In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("1. Loading Craigslist Data (Sample of 10,000 records for performance)...")

df_nlp = pd.read_csv('vehicles.csv', usecols=['price', 'description']).dropna()


df_nlp = df_nlp[(df_nlp['price'] > 500) & (df_nlp['price'] < 50000)].sample(10000, random_state=42)

print("2. Text Preprocessing...")
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) 
    return text

df_nlp['cleaned_desc'] = df_nlp['description'].apply(clean_text)

print("3. Applying TF-IDF...")

tfidf = TfidfVectorizer(max_features=500, stop_words='english')
X_text = tfidf.fit_transform(df_nlp['cleaned_desc'])
y = df_nlp['price']


X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42)

print("4. Model Training & GridSearchCV (Hyperparameter Tuning)...")

rf = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20]
}
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, scoring='r2')
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print(f"Best Parameters for Random Forest: {grid_search.best_params_}")

print("5. K-Fold Cross-Validation...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_r2_scores = []

for train_index, test_index in kf.split(X_text):
    X_kf_train, X_kf_test = X_text[train_index], X_text[test_index]
    y_kf_train, y_kf_test = y.iloc[train_index], y.iloc[test_index]
    
    best_rf.fit(X_kf_train, y_kf_train)
    y_pred_kf = best_rf.predict(X_kf_test)
    rf_r2_scores.append(r2_score(y_kf_test, y_pred_kf))

print(f"Average K-Fold R2 Score (Random Forest): {np.mean(rf_r2_scores):.4f}")

print("6. Final Evaluation & Comparison...")
# Linear Regression (Basic Model)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Random Forest (Tuned Model)
best_rf.fit(X_train, y_train)
y_pred_rf = best_rf.predict(X_test)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

lr_metrics = evaluate(y_test, y_pred_lr)
rf_metrics = evaluate(y_test, y_pred_rf)


results_df = pd.DataFrame({
    'Model': ['Linear Regression (NLP Untuned)', 'Random Forest (NLP Tuned)'],
    'MAE': [lr_metrics[0], rf_metrics[0]],
    'RMSE': [lr_metrics[1], rf_metrics[1]],
    'R2 Score': [lr_metrics[2], rf_metrics[2]]
})

print("\n--- FINAL RESULTS TABLE ---")
print(results_df.to_string(index=False))

1. Loading Craigslist Data (Sample of 10,000 records for performance)...
2. Text Preprocessing...
3. Applying TF-IDF...
4. Model Training & GridSearchCV (Hyperparameter Tuning)...
Best Parameters for Random Forest: {'max_depth': 20, 'n_estimators': 100}
5. K-Fold Cross-Validation...
Average K-Fold R2 Score (Random Forest): 0.6477
6. Final Evaluation & Comparison...

--- FINAL RESULTS TABLE ---
                          Model         MAE        RMSE  R2 Score
Linear Regression (NLP Untuned) 5873.287592 7740.030829  0.581773
      Random Forest (NLP Tuned) 5214.455284 7259.578139  0.632083
